In [0]:
#Streaming data for preprocessing

display(dbutils.fs.ls("dbfs:/Volumes/workspace/default/telegram_project"))

In [0]:
# create folder raw
dbutils.fs.mkdirs("dbfs:/Volumes/workspace/default/telegram_project/raw")

# create folder processed
dbutils.fs.mkdirs("dbfs:/Volumes/workspace/default/telegram_project/processed")

In [0]:
from huggingface_hub import login 
from dotenv import load_dotenv
import os 
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
login(HF_TOKEN)



In [0]:
pip install datasets 

In [0]:
from datasets import load_dataset 
ds = load_dataset(
    "leonardoblas/us_election_2024_telegram_distilled",
    split = "train",
    streaming = True 
)

for i,row in enumerate(ds):
    print(row)
    if i >= 4:
        break 

In [0]:
row = next(iter(ds))
print(row)
print()
print(row.keys())

In [0]:
import os
import math
import pandas as pd

# =========================
# Helpers
# =========================
def pick(row, candidates, default=None):
    for c in candidates:
        if c in row and row[c] is not None:
            return row[c]
    return default

def to_int(x, default=0):
    if x is None:
        return default
    try:
        if isinstance(x, bool):
            return int(x)
        if isinstance(x, float) and math.isnan(x):
            return default
        return int(x)
    except:
        return default

def to_float(x, default=0.0):
    if x is None:
        return default
    try:
        if isinstance(x, float) and math.isnan(x):
            return default
        return float(x)
    except:
        return default

def to_str(x, default=""):
    if x is None:
        return default
    return str(x)

# =========================
# Output
# =========================
output_dir = "/Volumes/workspace/default/telegram_project/processed/messages_clean_parts"
os.makedirs(output_dir, exist_ok=True)

max_rows = 500000
batch_size = 5000

buffer = []
written_rows = 0
seen_rows = 0
kept_rows = 0
part_idx = 0
finished_naturally = True

# =========================
# Main loop
# ds must already exist
# =========================
for i, row in enumerate(ds):
    seen_rows += 1

    record = {
        "message_id": to_int(pick(row, ["message_id", "id"], None), default=None),

        "content": to_str(pick(row, ["content", "text", "message", "body"], "")),
        "language": to_str(pick(row, ["language"], "")),

        "from_id": to_str(pick(row, ["from_id"], None), default=None),
        "post_author": to_str(pick(row, ["post_author"], "")),

        "edit_hide": to_int(pick(row, ["edit_hide"], 0)),
        "noforwards": to_int(pick(row, ["noforwards", "no_forwards"], 0)),
        "scheduled": to_int(pick(row, ["scheduled"], 0)),
        "pinned": to_int(pick(row, ["pinned"], 0)),

        "views": to_int(pick(row, ["views"], 0)),
        "forwards": to_int(pick(row, ["forwards"], 0)),
        "replies": to_int(pick(row, ["replies"], 0)),

        "date": to_str(pick(row, ["date"], "")),
        "edit_date": to_str(pick(row, ["edit_date"], "")),

        "ttl": pick(row, ["ttl"], None),
        "restriction_reasons": pick(row, ["restriction_reasons"], None),

        "forward": to_int(pick(row, ["forward"], 0)),
        "reply": to_int(pick(row, ["reply"], 0)),
        "reply_scheduled": to_int(pick(row, ["reply_scheduled"], 0)),
        "reply_source": to_str(pick(row, ["reply_source"], None), default=None),

        "chain_from_id": to_str(pick(row, ["chain_from_id"], None), default=None),
        "chain_from_name": to_str(pick(row, ["chain_from_name"], None), default=None),
        "chain_post_author": to_str(pick(row, ["chain_post_author"], "")),
        "chain_imported": to_int(pick(row, ["chain_imported"], 0)),
        "chain_date": to_str(pick(row, ["chain_date"], "")),

        "invoice": pick(row, ["invoice"], None),
        "media_price": pick(row, ["media_price"], None),
        "contact": pick(row, ["contact"], None),
        "location": pick(row, ["location"], None),

        "web_preview": to_int(pick(row, ["web_preview"], 0)),
        "story": to_int(pick(row, ["story"], 0)),
        "photo": to_int(pick(row, ["photo"], 0)),
        "video": to_int(pick(row, ["video"], 0)),
        "round": to_int(pick(row, ["round"], 0)),
        "voice": to_int(pick(row, ["voice"], 0)),
        "other_media": to_int(pick(row, ["other_media"], 0)),
        "media_ttl": pick(row, ["media_ttl"], None),

        "buttons": to_int(pick(row, ["buttons"], 0)),
        "emails": pick(row, ["emails"], None),
        "phones": pick(row, ["phones"], None),
        "hashtags": to_str(pick(row, ["hashtags"], "")),
        "cashtags": pick(row, ["cashtags"], None),
        "cards": pick(row, ["cards"], None),
        "urls": pick(row, ["urls"], None),
        "domains": pick(row, ["domains"], None),
        "identifiers": pick(row, ["identifiers"], None),

        # Theo mô tả gốc political là BOOLEAN, nên để int 0/1 cho gọn
        "political": to_int(pick(row, ["political"], 0)),
        "cleaner_urls": pick(row, ["cleaner_urls"], None),

        "toxicity": to_float(pick(row, ["toxicity"], 0.0)),
        "severe_toxicity": to_float(pick(row, ["severe_toxicity"], 0.0)),
        "identity_attack": to_float(pick(row, ["identity_attack"], 0.0)),
        "insult": to_float(pick(row, ["insult"], 0.0)),
        "profanity": to_float(pick(row, ["profanity"], 0.0)),
        "threat": to_float(pick(row, ["threat"], 0.0)),
    }

    # =========================
    # Keep criteria
    # =========================
    has_content = len(record["content"].strip()) > 0

    has_network_signal = (
        record["forward"] == 1
        or record["chain_from_id"] is not None
        or record["forwards"] > 0
        or record["views"] > 0
        or record["replies"] > 0
        or record["reply"] == 1
        or record["reply_source"] is not None
    )

    has_media = (
        record["photo"] == 1
        or record["video"] == 1
        or record["round"] == 1
        or record["voice"] == 1
        or record["story"] == 1
        or record["other_media"] == 1
        or record["web_preview"] == 1
        or record["buttons"] == 1
    )

    keep = has_content or has_network_signal or has_media

    if keep:
        buffer.append(record)
        kept_rows += 1

    # =========================
    # Write batch
    # =========================
    if len(buffer) >= batch_size:
        batch_df = pd.DataFrame(buffer)
        part_path = os.path.join(output_dir, f"part_{part_idx:05d}.parquet")
        batch_df.to_parquet(part_path, index=False)

        written_rows += len(buffer)
        print(
            f"part={part_idx:05d} | "
            f"seen={seen_rows} | kept={kept_rows} | written={written_rows} | "
            f"path={part_path}"
        )

        buffer = []
        part_idx += 1

        if written_rows >= max_rows:
            finished_naturally = False
            break

# =========================
# Write remaining rows
# =========================
if buffer and written_rows < max_rows:
    batch_df = pd.DataFrame(buffer)
    part_path = os.path.join(output_dir, f"part_{part_idx:05d}.parquet")
    batch_df.to_parquet(part_path, index=False)

    written_rows += len(buffer)
    print(
        f"final part={part_idx:05d} | "
        f"seen={seen_rows} | kept={kept_rows} | written={written_rows} | "
        f"path={part_path}"
    )

# =========================
# Summary
# =========================
print("\n=== PREPROCESSING SUMMARY ===")
print(f"seen_rows           = {seen_rows}")
print(f"kept_rows           = {kept_rows}")
print(f"written_rows        = {written_rows}")
print(f"finished_naturally  = {finished_naturally}")
print(f"saved_folder        = {output_dir}")

In [0]:
count = 0

for i, row in enumerate(ds):
    count += 1
    if i >= 10000000000000:
        break

print("Sample size:", count)

In [0]:
display(dbutils.fs.ls("dbfs:/Volumes/workspace/default/telegram_project/processed/"))